<div style="background: linear-gradient(120deg,#0b1f3a 0%,#173f73 55%,#2a7ab8 100%); color: #ffffff; padding: 26px 30px; border-radius: 14px; font-family: 'Segoe UI', Roboto, Arial, sans-serif;">
<div style="font-size: 30px; font-weight: 700; line-height: 1.2; color: #ffffff;">QCC: the public frontier + a chemistry refit that runs right here</div>
<div style="font-size: 15px; margin-top: 10px; color: #ffffff;">Google Verifiable Quantum Circuit Compilation &middot; the refit runs in this notebook &middot; every file re-verified with the unchanged official verifiers</div>
</div>

This version takes the best public file of each task (2026-11-14 evening) and runs **our chemistry weight refit in
the notebook itself**: starting from the best public DF-THC weights, a generalised reduced-gradient refit (prvsiyan's
method, our numpy port with a full Newton restoration) lowers `lambda_eff` at fixed angle words while every accuracy
gate of the verifier keeps a margin of at least 1.5e-9 Ha. Loaders, words, invsqrt and DQI are unchanged public files.
If this notebook helps you, an upvote is appreciated.

| Task | File | Built by |
|---|---|---|
| invsqrt | `1ba197b0` | manwithacat (James Barlow), on the public 46-qubit line (Syed Asad Ali, Ali Haydar Özdağ) with our dead-write fix |
| DQI | `821bf8d1` | Areeb Ali's 241 retained products on Syed Asad Ali's v12 decoder at extractor cutoff 18 (Ertuğrul Özer) |
| chemistry | maoxin123's v27 table, words and loaders `71735b23`; weights from Atakan Aldemir (`f4f2db44`), **refitted here** | maoxin123, Atakan Aldemir, Heliosli, prvsiyan, Lera (marculera) and the authors they credit |

## Acknowledgements
Thank you to **manwithacat (James Barlow)**, **maoxin123**, **Atakan Aldemir**, **Syed Asad Ali**, **Ali Haydar Özdağ**,
**Areeb Ali**, **Ertuğrul Özer**, **Heliosli**, **prvsiyan**, **Lera (marculera)**, **Alex Chilton**, **Kirill
Tymchishin**, **SomeJoint**, **busyaprime**, **yoyobar**, **mrmeows**, the team **i want to run gemma 9b**,
**jday96314** and **Oleg Sher**; and Google Quantum AI and Kaggle for the competition, kickmix and the verifiers.

**What is ours here:** the chemistry refit (run in this notebook) and the invsqrt dead-write fix carried by the public
invsqrt line. Companion notebook: [QCC 27.30 | What Works, What Fails and Why](https://www.kaggle.com/code/asteriospanoras/qcc-27-30-what-works-what-fails-and-why).

<div style="background: #fff7e0; border-left: 6px solid #ffb703; padding: 12px 16px; border-radius: 8px; color: #1b1b1b;">
<b>Getting the submission file.</b> This notebook rebuilds, refits and verifies everything when it runs. To get
<code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px;">submission.zip</code>: <b>Copy &amp; Edit</b>, set
<code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px;">WRITE_SUBMISSION = True</code> in the next cell and <b>Run All</b>
(about 40 minutes on CPU). Please keep the credits.
</div>

In [ ]:
WRITE_SUBMISSION = True
REFIT_MINUTES = 22
import sys, os, pathlib, zipfile, json, time, subprocess, hashlib, math, csv, io, base64
import numpy as np
T0 = time.time()
def log(*a): print(f"[{(time.time()-T0)/60:5.1f} min]", *a, flush=True)
inp = pathlib.Path("/kaggle/input")
KM = next(p for p in inp.rglob("kickmix") if p.is_dir() and (p.parent / "verifiers").is_dir())
VER = KM.parent / "verifiers"; sys.path.insert(0, str(KM))
sha = lambda b: hashlib.sha256(b).hexdigest()
SUB = pathlib.Path("/tmp/sub"); (SUB / "dfthcsos").mkdir(parents=True, exist_ok=True)
# Find every public file we need by its SHA-256, whatever notebook carries it.
WANT = {"invsqrt.kmb": "1ba197b0", "dqi.kmb": "821bf8d1", "dfthcsos/submission.npz": "f4f2db44"}
LOADERS = "71735b23"
got = {}
for z in inp.rglob("submission.zip"):
    try: Z = zipfile.ZipFile(z)
    except Exception: continue
    names = Z.namelist()
    for n, pre in WANT.items():
        if n in got or n not in names: continue
        b = Z.read(n)
        if sha(b).startswith(pre): got[n] = b
    if "loaders" not in got and "dfthcsos/angle_k_0.kmb" in names and sha(Z.read("dfthcsos/angle_k_0.kmb")).startswith(LOADERS):
        got["loaders"] = {n: Z.read(n) for n in names if n.startswith("dfthcsos/angle_k_")}
assert set(got) == set(WANT) | {"loaders"}, sorted(got)
for n in WANT: (SUB / n).write_bytes(got[n])
for n, b in got["loaders"].items(): (SUB / n).write_bytes(b)
log("public files", {n: sha((SUB / n).read_bytes())[:8] for n in WANT}, "loaders", len(got["loaders"]))

## Our change: refit the chemistry weights here
The refit code (our numpy port of prvsiyan's generalised reduced-gradient refit, Apache-2.0, with a full Newton
restoration step and Lera's gauge helpers) is embedded below as a small zip. It only changes `W` and `H2_I`; the
verifier's own math (the competition's `dfthcsos.scoreinfra`) checks every saved point exactly.

In [ ]:
PKG_B64 = "UEsDBBQAAAAIALMGb10u2vnHkQIAAIYFAAALAAAAY2hlbWV2YWwucHmFU+9v0zAQ/Z6/4tQvSVCWrQgGmlSJqWtZBVSlZUwwTZYXX1qriR3ZTruC+N9xnB9tpw3yIUrunt+7d3fu9XqfZUIzuBqffLseQo5G8QRwQ7OSGi4FbLlZgVkhbFDxlKPyNcitgJzaeCAkrHmyzvljGHs3GnUH06csNatES32qE6mQi1RRCFyQVIcjUJiiQpFgCFQwJ1LQZE2XyGA8nFzdfJlBoFcUkhUma2Rh3Ov1PJ4XUhnQOx1BWorESJnpNirKvNgB1SCKNiS1Z7FxYSVjLjQqE5xFNhqj2HAlRbxEE/hfh0PyfTSfjCej+cKPwO98+GHopUrm0PqJD/w0Gp2TStr+5JK9eCbet6A93rQbSZ2iYpkh2UrFrEW9y+uhkIqQZHKLymu7M3jWR5P1Q5AKtFFBXVE8uxx+uvw4umoBoTcfjcl09vMFniZbdcMPvQ9ds+NMlSShdipBP/QYpnv/QXjhgX0aRTc4kiac2bkEnWyNMKUSLTCTlJE9S4f8l6he8dToVlGvrItDtpQ/IiMtyGGmFvLu3H0u7KcoYqqpUnQXPNdlu3p3/rXN1L/+fQTTMAJmdgUO7NnUypjzNzX1AxpK+pbUReujfTLx70M4gbP4LbxqMvagUdR6XIRHjVhEDYfz5hyI4ldQre2gGUTjlNWlVxCXPqL5vb5wvpwrdre2+qndgjVwASxOeYb6j1MQ9npv0BIomuuGeXvclDp3599W1p8avk2eB1+/rlwf4W0HtrEu84A+cj3oH9W7jSyTK6i9BEGz+C4TgVGIBNkSbehBSTN43xTbNczWcTTlhvnlS/VEoSOKDve4UatekaP9/7MvdXBYNQpUyx0piwLVoNnQm9lsNCej6Wj+8Ufo/QVQSwMEFAAAAAgAswZvXe+O89cADAAAqR4AAAgAAABtb2RlbC5weaVZbVPbSBL+7l8xlXywRGSBvXt7V+w6lcCGZKsSjoAP7srlqGRpbAv0xkjGQJz/fk/36GVkSG7vzlXY0kxPd0+/PjO8ePHi3b0flOL3k8Hkw7FIZKmioBDZQlgbR3wYeX/Ywi/FyR//fPe78NNlLMUmU2HhiE1UrjDixw9lFIil8sNIpmUhrHSd5A+OWMSZX/7ys+32en/fpCLIQimsYCWTYuXnUhTlOnywXXGSqWQd+4XwlRTlSoo7qaJFJFUfrMJFuQqKrHCLIFMyShfKd3nMS/xyZf/KC6RSmfJGPXmf+2kRZSlGVbZernj2vfITAWoV3Yv33nzeF2Nhrb25cAW++/aXETYRis8lxs/ERJx9mYhFFsfZpuD1+XoeY39KLqKyR5tg43yUyhdW4qtgHePR0fPeRkbLVVm4tP+3uY/dDkbugc0ScnVXRA9+qo0QR/NdKt5Nr8jjqCQRJLziJ6K0zESapYNULv0yuoNaviJbfxyKUqqkEHGUSl/Zjeq1NFhRq7ZUSwiEMy59FfnzWBaHYiOsc0ccOeLYxu4XEZzbv+rDtUc8YYxSIPRdcaqFZ4tFISH+KgABiAeiWCcw6cbtCXyWnprT75iHA5B5KhAbjAY8vxr2hZ5/e1CtxQJeta6/vkyY9J239HMmDQ7EKyLdxn6y1asEHjF44P6F366CLyNe9NFP5qFfydf0r8Rw/2ctKRCW1nbLGtHcdnOE8a1dra8CitbfqpKpVSG2RI/geE8vG6/YbhE6AzHSXEWpLD3/ufSUot3axHk7ITrb4DsE3+2WV13TDrw8sq5tcYc3/IHBgA203XonWGPR+JiDYRHdy1BkqRzMs/BBKLjZ7r148aK3UFkiPG+xLtdKep6IkjxTJYIuzUo4LEuLXjWUNU+cpAJpl+Z6fRFECI8sL6MkepQ1Dx1WHinrF0W0TBPkeM2D4lje+TGxCaRm80zG1rzaxKUFYdLrHb89/vAOuwO5TO8ilaXuUpZW//PxscdziMX+fpnk+xzDAaVK3+71egEKRiE+IRvjQzZtKBewQJQi0D2rkPHCqcuUDJcSPzrO5yorx39zBHPybuTD+BT2dNiW/GhrdvSBZZnFoVis41hYp3Dn+R6YnA6GtigphX7lhcIiq2WohPahKDdZ4x6kbRgFKBgW5y/58Pxg7+iAVzVykOosZ3p6OAPfTKwLaBShTqhVBpVtgVmpUFAzhVwO1wGigGto4ZL3az4wEaWJdPEglUwDadnN5AUYydLXBMUqWpSFMXuK8b/+0ryS/dxTR/+eV79HoDnVdmwI154Py4zhSwgNEGelWgelt/buZFBmqrDS3PULXyn/weI92mJfWEPx22/CIlcg1Ie27QiDjN1l209E6N99IkVM+vHSTdE4LB52hH8fFeOhI26kzMMoKcYTtZZ2d0N3w5rN9PB01goYNcNwQDMcLbRvI2QIFKXYaEOjXTeaGrpzRs66UtdMBfOwx6pye7qjGmrbWPdLS/svyUL3H2dn7869d6fvzt//C4bC6LQvKan6M0Sifo1hZ2RNgaE9dnDLOJMRmJra0QqM9mds8Lo9NwsWsU89UGvdKgzGXY3LZxmX32fMHN/SFkmnAdfrPaJEeUBhsfq5UreD1/ktsh1s7IbkmT1eYOscyJrBg7R2bfme+jvv5Q1vyZ3YYm9PjBqqnArQmD1qerspCV03ozQl/o0MI0QzlyRUFARb6WU3O0Fm8F70vzLtt/3b0vvacP7mpvlj3xTK9AQOIIaeXeZdWPRsdxV51EaHcUM93ZnlrX8uq1ydUGw+TvufS/KJDiy8Tkb9WbtMxsVOSE+o0cio8f0prIyU55+uOBheq6M94Cr/TsaW/YTbpGrPe4JKQbYGVrEuamqH2FCTbKbw7ohm3u7yO6skVkGT5E6CqEk4bMjR+tt+ErlPtZ+csWpvxJk76UxUIPANKJ6zLiarzeD5FagRW0/p2Pja5lCXdJ1gyWRnN+hcYuLsCqpCousW+hAnWOWRXe9A9Ljx+GQ0ruTuZMLnUFusKP3gxppWC6aK7HJoKcJEtrbRk6GZbjbUhRRAP4xpz3bzbF4Xi/c71Y2+0aSJ8qUY/N+fpr8zfBrV3d1IjwrDEhZ1WZFm5mpe2eBRqqyw6pign2PDIU8223UAuHzPasfG0PGMNJiqtgO8v2pshKAiZZ7/vATcJrWOW5XeJlyNscalSAWnH6w97q492Q1AZrbHPI1dfzTJ6rQCInbUPHRUGAxe01HAqYOJn2t8qGufUUxGxS3YnTAg3gPrV3UyGCQdtQhZJ/69RSsd5NWBme8hbeFnMMK+32hb/CfLQbc0fhBh5C8Jiol5nAU3BYGpsGVbqdkmRXjyJz37o3zQbAda444tKzvCqH3Tju1SJYHZU9jGqbgAH5EBtWUcMZSDnw5gml6bBQDca7+UdR7QaQ19V8rQozM4OwaVNS/GZNMOnD1nYYDeUcAnzPqwTw3IQulpmNjGcf5r6ifyEEdxSAo3R/Y3MoXhVMgC8MIpzaFzF06k+OKrAYK7zfEUgH+xTgM+jJDo7WZLBpGCj7taaAfNfj+nT+ucOm3TvDmHbjjeNRxsrbzstg5FIc5ugVeugo5DVsOaPZ9NO97Eqhx/NVZZVi5dt7/d9OgyrVoHnl9hwGwdOAQ64rLq7xrc0rkfpAacu6tA17ywajj7Rly2BLcRDg2EzJ49slmD7K6lzeGTyiRJXj5YaPJh+ZDLMUZwVDHBG5FObyMqbVFkauwRB/xOiWJmBHTQNfa1c50717XNeKFTA/LmoZV3jl4CBsSGzsFtoA3N6mEeAWiFUTp8uhTQKU4lht72qosCJAWe8W1To6W31+KAcVBtWdAZbQ4jjVBiy7FliKKriXGD4F/pBYNqQUvfYKBOTUbMEiw12CWthze2GccaIvH4kVG+eZ8skwSMnkhIdgVUVabWueqnRvADhdFpDuXBqu4qxpTc1X3ImNbzhcy4Tvd56I8/mjZDHoMDGWZPFzLSckADHSnTfsyLPblY9GdP+gJVA/YRsWt8pGn6qZ/2W2ZUtnA+a0pXt3NX5RUSnS7gf2lUuI2r3NKlS5dm+pOnfW9d0j4oevZ1YD0TN5RiHDhQGRlpQMpPlIQmTm5m7s7vdrIkd/JbTpPrvpEeFNvPJMklxd3l9NDh9JyJH35eChQTmeojOd2E4jQS0j0D3VBcG8pyilnWJfkOCvJucpfuozjFYALuSsOmK/WMDrgQd5HvrYbWJ8DT8CrwcGKCxB0kteSYHXSqKu+cqyrX47qUEp/nyyrvKTzZD5fdMwlfb7SiddZ0qjfVfOrESyq4G/tZYxFj5GaZlX7cRewbjuyEzH6ow4lOpFfBtHrD8AzGgh7myHPxSL0UZG6Q5Q+WYcbH8y5adTooNfR0wakN7elyw1FGfcykZF82hOSzx3NznurwarjDrEuEpPGocX7ty1GfAAClf62cF0c3ksoRcFcfaoGA1cObHPILyaQsX+lXLRBDCV6Tb7sAiDOUJfZ6PYomxgUWo5sqhipKaBBkaQAElBIKovsXhGWUwMmgPrDb06UxM+hOPTleGR+T39EPGHbmZnal9jrVit87+oqu0r1FMy2WSa/A8QOG+FBCKLM6mPAsLJli7n56mF7Ndg5YBHbvp+nVIRXY56b1LTtYrDQPTVeRU1586KzR/IzZw51p0/6kmFMxb25iL2jTZyqbxzLROwaW+1TDyx3sd09lxmRji9eARxo1UpLlBLz4vxBjUBAOSBmjVohS/+sjW5h3/RsVlYgIkpXTFlC09T9EGFR+74o4oQvkrqfow2XnU1V+9H+rxh3a76Lx+z+DxM2t0+2gETKNwDYNN5UhKqNU9jCz1GF8q5V2G4V2DwasiW00wMpyONLofzkB1ictFrlgJ5Epd6HIikYNDf5L2MLdX8v2WDD3f2O0P+MLwO/jme/BnYtdvPM/YgMipeb/ta1SdPq74Tt5KoZVydO1rqlyO62Oz0vkuOX0ptsEmD8Gtd06xSzctNVmYL6ER+bEkVF3GnnxcKM5zlXmh4FflF6ZWRp5JhdmSwJs2I0yrH5OHfA0dOi8NZztH40ZCuptk5PZ49Wrth3daIP9s12BCXv/BlBLAwQUAAAACACzBm9dhg8e5X0KAABAHwAABgAAAGdyZy5wea1ZbW/bOBL+7l9B5HCwlMpay0mLxj4X280lBYK0WzS5D0UQeGmJknWRJUWkU/vS/Pd7hnq35VyytwbiSENyODMcPvOQPjg4+Ca8lSu8QZBxLxSxYpnwQ8WSmKmFYHKV+dwVTGRZks1GbMquP/7LYmezgKd4OTu/ZAaPvaLdIZFzjcEYAiXzMPakafd6H1m8WqYbliYZVPsszR5kuOFxX+bTzYIssNFunCbpJguDhWKj4ehd1c9iH1PuLsRgZA8nzOOKS6F6ZesvD/euO3gQWeiH8GQupBok/sANM3cVKjlwnOE7bjE38cQvzenMMYuTeBCLgKvwAc6mUajNI89/CDJDWr3LwW/nn66YF2bCVSHiUnS45AhZHIQ8hpnJv9EovDJsCg0US5lS8IoB3NWzuEksVcbDGMrZF/FDJXEvg8lJxrX6OXfvoEclrQXwk4wJuLhhKgt5hEhCgcXEGlqZsYrlMknQ3zNZU9cbzO2HLg1A9Ny7HqmR/AGWagXSZqcLslWyhcjEuFimMhck87NkyZYIXGRfUXS+Zsk8EksWwgfBPe1akrkLxlcqoWEW4z24jrVwmVzwVFB+bNg3i/1msVMTzcgWzlweRdrRwqAwDuzewcFBT084m/krtcrEbMbCpc4ZHseJ0j7JXiFS4VKUz7nZXLI47dUml4Oblvd6Z7NPH7/Ozi9///0b0vXIPnGO35+87/V6bsSlZJ++fRr3GD6e8GFIGCNdZoYUkW/lWq3cL7yNpkdicIKHoHxwpiMxeI9lSeXUEYN3EInldDS0WJQE0zRDyM1cO31IqX31FVY0LTSas5jtzth86D20hyM2wPztRtqM2H8N997AtK0+2J21Amer8esVGmF7W/z57DPEcKQthkcQ43vLxE/XR5DHqc2zjG+Mm8rwQ+aII6u2tfUOu/T77ZbHX06hbMvRy7Pz8yHEX5JY9Kq18gO3WKa1XgBLp7GcXmcr0Qh6AjnGFrG3xQOPVlwJY13012NrIy75Eg0a7JKbPt7mHp9poOjfWiTSUNi/rQcMS+3azEruQ0x6Dpkxwhc0YQUgMNkvhWRoQjK03+rnapzrtqKJEKEd8xaITFbkMugqH6tmp397a96Mi0jWRoY+kE/lDtehoU8msPFihii6rtVSVHXzyBWj4cUon92kfzd9HQ6kXiHNhejYv226Wmm7yN2TCmhQuocBoulZobTxLvZ5tt9+C4Zb7CJPmF8B2anI1KZKH+StTp9GqhS6qrRuTFcNy9FWbGXehQXkFMAIlUQECQ4QYMnXoZq+b+i/oEBewCHqW0k/L0nqs1/xZV/X+QOgBBzGjIqOMLQ2s7125DQMcMoEpB3R3gvnPJINTNFOoreLRSy9bDUWeULpF0UG/oXSJ0QURmZuTd4IGOYsTbGYnnJHaWrDAVLI55J0sX9QqF6mkTZ0q+M6TyEKyXK1NIBuWAm4RPFDGNEUhTGPAlsm0YMwPmNHY8o6sq8NW5dR8yTRAWq41Yyq9vDtEGsNN80iByVVNHcp1CLxahhDLsyWXN5pRNrNRmPNPjBAxU9mBNA5NOtc1GRgpgt7nY+kr6HFR1K6gmzGV4C/C7HlN8JXu7pGj9aQ5C4r+5epv7ZdsDbD1ENJYTP53zaT3xntBjHX2YBxwWWIOlh7kI8dDYekl7vToX3UcAik4Z98mYLRLENwuSRbMlCTVDKDjGAPHHQJ6qQJOsUVSjgomKZoC2pukDGY+4NnHjWGGRhcFggI56AVqppsmUjF/iAr/ihZHaIQeitiZCLTM49Bb1DJV+j5g286mBzxE858Tt15pmyiPeUEX06r8vHldNIogUctIABNfRESYIktjbDNBS5r/fM4cNRuvRneEglAdtOjxQaUJhPIR7V8dJtnD/r/jYmR3lMo0QMESHgTXQwWIgL9UzogfpQk2UuQAUzqaBca5pngdy3pcpUDQbHbI6nkvWEQwBabxrSvqfpX78ABJIA3JSJhwq12DCk/ysi1NiVRpxXhczuC3XBOH0oLDeqdeFRhPYomXHWoQqKP2AjjyMxzHpLMfBHm0VTmdlCRLn9nDjIL1G83kGBEsz2MSAPfFiRUThUk0PD7tGfZY6iekEWPsCpLVrFnZBY7Np9YJHyfPSY3B1FOnvB6cDu23/lPmj5QS06rSHrsP/W7GCD2xQ5yNIrwKq7hAtUpA3In8UwfefTiWvnxZ6ZPUNO3Q0pN+SNU7gJoMnROTiyNOx1c8TWRQaQb/DSUmqC2493ir2AndUz6LXZG1ox3FjzPxRIg1+01eb2xTXaE/VCGpMPibQh6HbxUmdLH4YqVJ9Uc/Zjf12W07/a10n4OvH2rAiLIwPZoulrjX7fTX7XLWxW/qxLW3jc4YF6/joddASlGev0tzbvs5IWxvppZ7DvlwQ3A+Kb2U1HC0ZHZpi8D0M2jdMEhdOhKJfZ5GNHJcG+Z0dtqq8zQ3cOU6Qi/EIgxoloaPD8LwAEdZ+tleCE4X7Z3RXBOSi67UHk/Xl9ctCh4q43whu57jIf9BPhhP/m8QFpozQ9tO1M6lGq9wXm75Z4agoJj0bLlq9sOB92iWGyjl4tATgrPiEKpjP+EqZHnhNnF2CkBDAlz7vXhbIMnmU9i8zQVQHGO13s2mKLrIdtsV5ar2a7Oe3YIpVezm4FzC33f9YNW/31LtuODQU7QJRE5QqY3zLdqz2BelzNzcmaz48w9ewM5x5rM6Ygq28uJMQMd9/siE9pWyQjnRL0C0OZtu5+3fugsq7nmYFKpGBhaSRrApn3btNBLh66u0JZ7FlTX0M+WJkfwtkGaPFNTKDNvG5pthwRKwX7FAIN2UbmDUJePCVuEyinfdlYukjVUjN52LM5adTKWN8XUh8zbpRhrAISPPwIK4ahuzFUdoLurisodBtPtY/MY6ytTy3wFnunnBOyYsKkwKl876mFUhRCVkNFtrENjyturDo+r+BGrmHTw1cKuhuKRnuo5Q2nWD9Nq2u5Zi5XaOSjXK31Il4DHXWkHi2keUrGrvEEGns9h+vyJQ4g2sC4qk7+oktDntbX7WUZQH0p37f+TbjfoNMquJtPljyr1IZXlDE3/0DAp7548cOi77Amnt0e3yZ7LD4arMF7tXsLo1d5d5JwLADOdlxiJlcHZVnD6EYAG0kkPhglFN8ZJtsE5nNQ9aq1PZod9Fd7lSa+7f2AdJz76dO+irVSc7MWzvfHYJUD0CUCqL/C3XCVEI3RiLFdtXk4t64nm52vVxsXXZ4LUqT+A1gmqecEIDCNPvUTnHh6NIKkFiWl2Vi5CFrRvEMvGqbLYSHRPYkhzV7bpALKrigxQOf1evW2ahYzqfflzwfjW0iW+fn/uRMpypolHzTHhWQfy/P+H1DJhHxsseKCGY3voP0lg/6OP46d4Yj/T4Kc+xTbDgqI9tkdozRPrUf8b2w4k/c48zROgH2ui+kg3cvRkS1Q+E4diYOhjAaaYcd8Z+T3OyGKkG4rrfpI6MPj5Wc8CPUb/NEEjTkiP09DjlNr/1wm8sWTlgVrvUr1+jTP1nqsF+qzbt4bNS8r1vmqtV7yYzygUtC8l92t8Frjb599ihrbd++et733LW8v/AlBLAwQUAAAACACzBm9dokPaM7oGAAD4DwAACAAAAHJlZml0LnB5tVffb9s2EH73X8FlD5IyWbOcpGuzqmgztN2KoCvabnswDIORKVuLLGokndgJ8r/vO1KyJNvDsIcZrUIdj/f7Pp5OTk7erBcrURoxH17zheLlIuclUyLLDZMZ8+9D9vN49kvAuGFZvhFzdi/VXF+yVV7mq1wLVvDVzZzPRJYxvb75U6SGGTkQSkk1G7OXCfs6Dpl7je1rHLK3swWv2KuEZYWUivnyvmSpnIsfmVkK9uaaZVKt1gU3uSyxLgp5r9m1UNzTzrYZL6JqGw7eVDxdiuE4GoWMl3N7vJTlsBQLHL4TTFcFPPmOfRT3RpJj2kjl5MrSSHuAp5Y1laU2iuel0bVOVqk7nW956emBU7tQC9LLWr1BdHJyMsiUXLHZLFubtRKzGctXlVQGJpXSWG16UJNMvhLNulyvqi3jmpWVk6DTvNpGsgJT/iAaKS7UD8LxrBCnotn6Qu59UvKmEKvQORuydWkXg8G7619//cwSdha9iM+fv3g+eP/m61u8j6LReDAYpAXXmn0mxy4HDL+5yOAEtJnZzNeiyEKnDZKXvBJ4i5OxGL7AYtwsFsmZXYhKJ7EYPgtZIRdJpRDGwEmlHwmLvnyC7q7Ffld60Ge2NPDbv/0tgwpyi3G9yApwWueGsDHcLbHvYvAdDO0Lgb04g2efDONBxnOwi0jFldF1ODbW0Y5jMmR0oPYvEne8WHMjfDB+VWvh2Hfc1xxZotJPmJx4165zbLK8aUgk2xjedHdA8Tl46cQp88d44AwcA6EVmmcoecN84n3FRh3jrASBkizZR1m2Ucwg82KEKoDAsiKf6XArcU4MsTgjpX6rEa+LiWcNRECdUQvrhjcN2Pdk7E4ENROEQDpXim/9SS0OPtbQACHDJo1ByPb3485+3O436XYWTTs2txq14eltoxEGijFFt32N6XW4e7cOdSTVEUNmkfA5/pPk0Cloq4IXMy2LO9EUBgBIro1QyRlWeVlidT7CUi1lEhM8bRPKQVijTeK78mifQSdxW+fIg1BS+2cBpXjLcm2zyEQB1KXAahdawBFglJvWA6joxt5pbLc32NyMolRWW7+lmhHIhE4RPToblRJ3s7tcFk5mXmZtIQG8b+Eto4tD+DYAe/VHscrWpf+wR7eRbjrHtdhDuGvN4IAXIVBNCA4ldfIWixht30RvVuS3AsoPTnTSS7mFJeqApxKl83nFN/lqvfJHNpEofmQVxZNSSyC6BweBAjiYgdFmxqeixHGfBJ6esjHK19+6s4ElUPvY/obgIDi0tvYNcslmyCVJr2H3Xq9TBzTXhY+wW8T6k6eJq7OVMEs5T7zr4dW791+GV17IbuS6nOtk4sMzim0whRWbSENAeDTMTfwqe60ljx4FB3n3Ll3Zh4woqSQC9YKXGVlgjcshPsfrYvf6wrEWGq8Xo6e+11Sk8Cfa9KiHWevUD1xFgvpi6rrdpQH6/E5CU5cEHNyP+va/Zx4latUBhKPxBRjazqHZpN6jS/JIK0CsTZ3vFJwTYJ33Teo2Iv3pbTa3l595mJ8eb5+QDsMeKYS4z5+YGLNHQN6pnJzUAHwyvYyeZdiJ93biZocg/hFkeysR8QcQveNVkdE9YLndtUbs52AvaDAkcjsnNuK37BERVlSAALGz4MlG4RGPy8UT8x87YDQ0o8tolD3pwAuOxvwlxfXCxvmWBsszu6y9x+7FYchvlOC3+6iPGtpa0G6R3s2Mop0AbMFbnEcp09ATjzvghnGwHja1oSHDFxvCY7SeylMdYOy0FmNImKPpD2dPVLgSAq2uco4hSeNAsaWhzuSoo6LNendeXcoCk3luljTQ5gomqkVeajub7jIEsEYsdmht3dhD5X8dZw4aLO3d8sevdzvaHLnW2+lt2Bl+pgcZJvlFgWbF5wNCfuwm2WVPOkt7HB96c8FuImhngSNTAP1gHU6maFpMC8Mf9pXa37esFFwNj3xDcCXYUiAv+HRyWTHIiuhjhs11wvyNHd1wp5ttBeTu3+fWB5rJPkxgEgE0HetDfw1XhP2Ea5aRYA3MI8CWI2BB5Rr3vUSZkuws+oorhebBvOTFIiq00X/50Pva7qEr8A8Ozu0gE0xG0wPA3gfMDeUa4ltXkJ7ZPxTYO47BZq/C/tfq6hXNjZSFv19oAb6SCAI0vxOzsnrwK26W4OVkKD6N76/qWqQpfZ6nxqctp+FzyK5C9hM27qP2I2Y+8f6AkSA2r/R57ShXloR6B/U3b+pOTS7HU/YNKoTkdSofRYYx8HeET7wl333vN/ethEtEr7hJlzVSUuHD/Mb201MM+n8DUEsDBBQAAAAIALMGb116XiG8vQEAAIcDAAAHAAAAZ3JnMi5weY1SwW7cIBC971eM9oQjYnlXilS1dZVTVvIhh2hvVRWxeLBRMTiAE+fvO+BVdt30UA5mzMy893iDHkbnI9hpGN9BBLDjRnk3QOc70Evu8HTYSCNCSNGe0af4ugFa2+2WfuBNxx4EqMkYeMS36Cx4DNF5ETXFrBHSnbRIp9IN4xSxBRSyBx1xqfkGsUeQvfMtvKIPqc0FqY0REQP06LEoiS2ztqjO+MgCGsVh5oBj4NBwUB6RQ3Sm3uPtruIwiFnH+stZcVr5BOq0sxxz2O2Lj7RyHp5Bk1phO1wqrrpzCQdJlDsOHZE2hJV0lKqTbJFSrMo9FUi4XYqOh+MqqRVYR/6PpTCG0aaD0pacYb74izZjYZy8TTe+iHgQJuAn0LFMF0y4p5Cw4Huy5f8Qj35aAwqZHJvhB1Sr82GiY+Iw2grTlSbE8MIYWXKTWorySBYtMQcvnW3rR2ex+Fn9WhvqPzyk+T0PIvxOTnZkGjHck8drRxtF9ZlF+VViXuSkoQ3TwKqSXsBMKA0ZTzgXpcGZV2SEc59zpK64egJnIz4PljR50YY6W35p+JeHJ+fyPK+mcP0I8kDuKroCTaXY/AFQSwECFAMUAAAACACzBm9dLtr5x5ECAACGBQAACwAAAAAAAAAAAAAAgAEAAAAAY2hlbWV2YWwucHlQSwECFAMUAAAACACzBm9d747z1wAMAACpHgAACAAAAAAAAAAAAAAAgAG6AgAAbW9kZWwucHlQSwECFAMUAAAACACzBm9dhg8e5X0KAABAHwAABgAAAAAAAAAAAAAAgAHgDgAAZ3JnLnB5UEsBAhQDFAAAAAgAswZvXaJD2jO6BgAA+A8AAAgAAAAAAAAAAAAAAIABgRkAAHJlZml0LnB5UEsBAhQDFAAAAAgAswZvXXpeIby9AQAAhwMAAAcAAAAAAAAAAAAAAIABYSAAAGdyZzIucHlQSwUGAAAAAAUABQAOAQAAQyIAAAAA"
P = pathlib.Path("/tmp/refitpkg"); P.mkdir(exist_ok=True)
zipfile.ZipFile(io.BytesIO(base64.b64decode(PKG_B64))).extractall(P)
os.environ["QCC_VERIFIERS"] = str(VER); os.environ["QCC_CACHE"] = "/tmp/refitcache"
sys.path.insert(0, str(P))
import chemeval as ce
from model import Model, split, unsplit
from grg2 import GRG2
base = ce.load_npz(str(SUB / "dfthcsos/submission.npz"))
w0 = base["W"].astype(np.float64); wB0 = base["H2_I"].astype(np.float64)
def metrics(ww, bb): return ce.evaluate(base["angle_words"], ww, bb - ww.sum(1), base["tree_edges"])
m0 = metrics(w0, wB0); log("start", {k: m0[k] for k in ("lambda_eff", "error_1", "error_2", "E_gap")})
best = {"leff": m0["lambda_eff"], "W": w0, "H2_I": wB0}
deadline = time.time() + 60 * REFIT_MINUTES
class Stop(Exception): pass
def on_check(xe, okr):
    p, n, hp, hn = unsplit(xe, w0.shape); ww, bb = p - n, hp - hn
    m = metrics(ww, bb)
    ok = m["error_2"] < 0.02 - 1e-9 and m["error_1"] < 0.02 - 1e-9 and m["E_gap"] > 3.914898 + 1e-9 and m["gap_in_domain"] and m["Lambda"] > 49.0
    if ok and m["lambda_eff"] < best["leff"]:
        best.update(leff=m["lambda_eff"], W=ww.copy(), H2_I=bb.copy()); log("better", m["lambda_eff"])
    if time.time() > deadline: raise Stop()
M = Model(base["angle_words"], base["tree_edges"], *w0.shape[:2], cache_key="kaggle")
G = GRG2(M, w0.shape, log=lambda *a: None, m2=1.5e-9, mg=1.5e-9, m1=2e-9)
try: G.run(split(w0, wB0), 100000, on_check=on_check, check_every=50)
except Stop: pass
log("refit lambda_eff", m0["lambda_eff"], "->", best["leff"])
arrays = {k: np.array(base[k]) for k in base}
arrays["W"] = best["W"]; arrays["H2_I"] = best["H2_I"]
with open(SUB / "dfthcsos/submission.npz", "wb") as f: np.savez(f, **arrays)

## Official verification of the exact bytes that go into `submission.zip`

In [ ]:
def verify(args, timeout=4 * 3600):
    env = dict(os.environ, PYTHONPATH=f"{KM}:{VER}")
    r = subprocess.run([sys.executable, *map(str, args)], capture_output=True, text=True, env=env, cwd=str(VER), timeout=timeout)
    try: return json.loads(r.stdout[r.stdout.index("{"):])
    except Exception: return {"status": "ERROR", "stderr": r.stderr[-2000:]}
res = {}
res["dqi"] = verify([VER / "dqi/verify_submission_dqi.py", SUB / "dqi.kmb"])
log("dqi", {k: res["dqi"].get(k) for k in ("status", "score", "toffoli_count", "num_qubits", "value_errors", "phase_errors")})
res["dfthcsos"] = verify([VER / "dfthcsos/verify.py", SUB / "dfthcsos"])
log("chemistry", {k: res["dfthcsos"].get(k) for k in ("status", "score", "toffoli_count", "num_qubits", "lambda_eff")})
res["invsqrt"] = verify([VER / "invsqrt/verify_submission_invsqrt.py", SUB / "invsqrt.kmb"])
log("invsqrt", {k: res["invsqrt"].get(k) for k in ("status", "score", "num_qubits", "toffoli_count", "worst_error_over_tolerance")})
assert all(res[k].get("status") == "PASS" for k in res), res

In [ ]:
REF = {"invsqrt": 2076774.23, "dqi": 24439463583672.93, "dfthcsos": 3355599588.32}
r = {k: REF[k] / res[k]["score"] for k in REF}
print({k: round(x, 6) for k, x in r.items()}, "GM", round(math.prod(r.values()) ** (1 / 3), 5))
OUT_ZIP = "/kaggle/working/submission.zip" if WRITE_SUBMISSION else "/tmp/submission.zip"
with zipfile.ZipFile(OUT_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(SUB.rglob("*")):
        if p.is_file():
            zi = zipfile.ZipInfo(str(p.relative_to(SUB)), date_time=(2026, 11, 14, 0, 0, 0)); zi.compress_type = zipfile.ZIP_DEFLATED
            z.writestr(zi, p.read_bytes())
json.dump({"sha256": {str(p.relative_to(SUB)): sha(p.read_bytes()) for p in SUB.rglob("*") if p.is_file()}, "verify": res},
          open("/kaggle/working/receipt.json", "w"), indent=1, default=str)
log("submission.zip written to", OUT_ZIP)

## Where each file can be found (read from the QCC Circuit Atlas)
The [QCC Circuit Atlas](https://www.kaggle.com/datasets/asteriospanoras/qcc-circuit-atlas) lists every public circuit
file of this competition once, with the public notebooks and datasets that hold the same bytes. The source shown is
one of them (the first in scan order, not necessarily the author: see the credits below for who built each file).
Files newer than the atlas scan (2026-11-12) are marked as such.

In [ ]:
atlas = next((p for p in inp.rglob("circuits.csv") if "atlas" in str(p).lower()), None)
rows = {r["sha12"]: r for r in csv.DictReader(open(atlas))} if atlas else {}
print(f"{'file':28s} {'sha12':12s} {'found in (one of)':58s} {'owner':22s} {'#sources':>8s}")
for p in sorted(SUB.rglob("*")):
    if not p.is_file(): continue
    h = sha(p.read_bytes())[:12]; r = rows.get(h)
    if r: print(f"{str(p.relative_to(SUB)):28s} {h:12s} {r['first_source_ref'][:58]:58s} {r['author'][:22]:22s} {r['n_sources']:>8s}")
    else: print(f"{str(p.relative_to(SUB)):28s} {h:12s} {'(newer than the atlas scan)':58s}")

## Credits (who did what), with links
- **invsqrt** `1ba197b0`: [manwithacat (James Barlow)](https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook), on
  the 46-qubit line of [Syed Asad Ali](https://www.kaggle.com/code/syed3000/vqcc-own-circuit-portfolio-notebook) and
  [Ali Haydar Özdağ](https://www.kaggle.com/code/ahoirg/qcc-submit-verify-and-package), with our dead-write fix
  ([QCC 27.30](https://www.kaggle.com/code/asteriospanoras/qcc-27-30-public-frontier-free-cz-fix-in-invsq)).
- **DQI** `821bf8d1`: [Areeb Ali](https://www.kaggle.com/code/areebbb/qai-fewer-toffolis-same-qubits-lb-27-30687) on Syed Asad
  Ali's v12 decoder at extractor cutoff 18 ([Ertuğrul Özer](https://www.kaggle.com/code/ozertuu/quantum-ai-27-083-vqcc-dqi-cutoff-17)).
- **Chemistry**: [maoxin123](https://www.kaggle.com/code/maoxin123/qcc-v27-4033-toffoli-coded-word-chemistry) (table, coded words,
  v27 loaders); weights `f4f2db44` by [Atakan Aldemir](https://www.kaggle.com/code/atakanaldemir/qcc-terminal-readout-and-phase-folds)
  (on Heliosli's and earlier refits); [prvsiyan](https://www.kaggle.com/datasets/prvsiyan/vqcc-verified-best-of-circuits-1106a)
  for the refit method (Apache-2.0); Lera (marculera)'s DF-THC line and the authors her NOTICE credits. **Ours:** the refit run here.
- Tasks, kickmix and verifiers: Google Quantum AI / Qualtran. Each file keeps its authors' terms; this notebook's code is Apache-2.0.